# Scarabaeus Tutorial — Units, Frames, and ArrayWFrame

This notebook introduces the core Scarabaeus abstractions for:

- Physical dimensions and units
- Arrays with units (`ArrayWUnits`)
- Reference frames (`Frame`)
- Arrays attached to frames (`ArrayWFrame`)
- Frame transformations using SPICE

The examples are designed to be lightweight and educational while following typical astrodynamics workflows.


In [1]:
from pathlib import Path
import numpy as np
import scarabaeus as scb

# ============================================================
# Helper: find Scarabaeus repository root
# ============================================================

def find_repo_root(start=None):

    start = Path(start or Path.cwd()).resolve()

    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() or (path / ".git").exists():
            return path

    raise FileNotFoundError("Could not locate Scarabaeus repository root.")

ROOT = find_repo_root()

print("Repository root:")
print(ROOT)


Repository root:
/Users/zael5647/scarabaeus


## Load SPICE Kernels

Scarabaeus frame transformations rely on SPICE kernels.


In [2]:
# Load tutorial data

import supplementary as supp

data = supp.load_data()

scb.SpiceManager.clear_kernels()
scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)

print("Loaded kernels:")
scb.SpiceManager.print_kernels()


SCB tutorial data up to date.
Loaded kernels:
                                 Kernels Loaded:
Source:   /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/supplementary/data/kernels/locked/locked_generic.tm   (META)
Source:   data/kernels/locked/ck/cas00084.tsc   (TEXT)
Source:   data/kernels/locked/lsk/naif0012.tls   (TEXT)
Source:   data/kernels/locked/spk/de432s.bsp   (SPK)
Source:   data/kernels/locked/pck/pck00010.tpc   (TEXT)
Source:   data/kernels/locked/pck/gm_de431.tpc   (TEXT)
Source:   data/kernels/locked/spk/earthstns_fx_201023.bsp   (SPK)
Source:   data/kernels/locked/spk/earth_200101_990628_predict.bpc   (PCK)
Source:   data/kernels/locked/spk/earth_topo_201023.tf   (TEXT)


## Units and Dimensions

Scarabaeus internally tracks physical dimensions and units explicitly.


In [3]:
# Generate common units

km, m, sec, day, rad, deg, kg, N = scb.Units.get_units(
    ["km", "m", "sec", "day", "rad", "deg", "kg", "N"]
)

print(km)
print(sec)
print(deg)


km
sec
deg


In [4]:
# Example dimensional operations

velocity_unit = km / sec
acceleration_unit = km / sec**2
frequency_unit = sec**-1

print("Velocity unit:")
print(velocity_unit)

print("\nAcceleration unit:")
print(acceleration_unit)

print("\nFrequency unit:")
print(frequency_unit)


Velocity unit:
km/sec

Acceleration unit:
km/sec^2

Frequency unit:
Hz


## ArrayWUnits

`ArrayWUnits` combines numerical arrays with explicit units.


In [5]:
# Scalars

distance = scb.ArrayWUnits(42, km)
time = scb.ArrayWUnits(3600, sec)

print(distance)
print(time)


42.0 km
3600.0 sec


In [6]:
# Vector quantities

position = scb.ArrayWUnits([7000, -1200, 3500], km)

print(position)
print(position.values)
print(position.units)


[ 7000. -1200.  3500.] [km ... km]
[ 7000. -1200.  3500.]
km


In [7]:
# Unit conversion

distance_m = distance.convert_to(m)

print(distance_m)


42000.0 m


## Frames

Frames define coordinate system orientations and origins.


In [8]:
# Generate common frames

J2000, ITRF93, ECLIPJ2000, IAUEARTH = scb.Frame.generate_common_frames()

print(J2000)
print(ITRF93)
print(ECLIPJ2000)
print(IAUEARTH)


J2000 (0 - SOLAR SYSTEM BARYCENTER)
ITRF93 (399 - EARTH)
ECLIPJ2000 (0 - SOLAR SYSTEM BARYCENTER)
IAU_EARTH (399 - EARTH)


In [9]:
# Display frame properties

J2000.disp_properties()


Frame Properties
Frame Name:     J2000
Frame ID:       1
Origin Name:    SOLAR SYSTEM BARYCENTER
Origin ID:      0
Frame Class:    1
Class ID:       1


## Frame Transformations

SPICE can compute frame rotation matrices and transformations.


In [10]:
# Create epochs

et0 = scb.SpiceManager.jd2et(2461809.72995654)

epochs = scb.EpochArray(
    np.linspace(et0, et0 + 3600, 5),
    sys="TDB"
)

print(epochs)


[8.86872737e+08 8.86873637e+08 8.86874537e+08 8.86875437e+08
 8.86876337e+08] [sec ... sec] (TDB)


In [11]:
# Direction cosine matrix (DCM)

R = scb.Frame.get_DCM(
    source_frame=J2000,
    target_frame=IAUEARTH,
    epoch=epochs
)

print(R)


TypeError: only 0-dimensional arrays can be converted to Python scalars

In [12]:
# Full transformation

T = scb.Frame.get_transformation(
    source_frame=J2000,
    target_frame=IAUEARTH,
    epoch=epochs
)

print(T)


[[[-7.63800639e-01 -6.45448876e-01  2.08119818e-03  2.69515025e+07]
 [ 6.45446450e-01 -7.63803475e-01 -1.76995367e-03 -1.39474218e+08]
 [ 2.73204101e-03 -8.58976692e-06  9.99996268e-01  3.83428230e+07]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]] [non-homogeneous], [[-7.19826694e-01 -6.94151054e-01  1.96064275e-03  1.77724084e+07]
 [ 6.94148447e-01 -7.19829364e-01 -1.90263423e-03 -1.40940738e+08]
 [ 2.73204378e-03 -8.58978435e-06  9.99996268e-01  3.83347425e+07]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]] [non-homogeneous], [[-6.72753446e-01 -7.39864478e-01  1.83164528e-03  8.51689094e+06]
 [ 7.39861701e-01 -6.72755939e-01 -2.02712302e-03 -1.41803815e+08]
 [ 2.73204655e-03 -8.58980179e-06  9.99996268e-01  3.83266607e+07]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]] [non-homogeneous], [[-6.22783574e-01 -7.82392324e-01  1.69476116e-03 -7.75421261e+05]
 [ 7.82389389e-01 -6.22785879e-01 -2.14288402e-03 -1.42059732e+08]
 

## ArrayWFrame

`ArrayWFrame` associates vectors with both units and frames.


In [13]:
# Construct ArrayWFrame

position_j2000 = scb.ArrayWFrame(
    scb.ArrayWUnits([7000, 1000, -500], km),
    J2000
)

print(position_j2000)
print(position_j2000.frame)


[7000. 1000. -500.] [km ... km] (J2000 (0 - SOLAR SYSTEM BARYCENTER))
J2000 (0 - SOLAR SYSTEM BARYCENTER)


In [14]:
# Alternative constructor

position2 = scb.ArrayWFrame(
    np.array([1, 2, 3]),
    km,
    J2000
)

print(position2)


[1 2 3] [km ... km] (J2000 (0 - SOLAR SYSTEM BARYCENTER))


In [15]:
# Frame conversion

state = scb.ArrayWFrame(
    np.random.rand(3, 10),
    km,
    J2000
)

print("Original frame:")
print(state.frame)

state.convert_to(ECLIPJ2000, epochs)

print("\nConverted frame:")
print(state.frame)


Original frame:
J2000 (0 - SOLAR SYSTEM BARYCENTER)


ValueError: Mismatch: AWF has 10 elements, but 5 epochs were given.

## Operators and Safety Checks

Scarabaeus prevents invalid operations between incompatible frames.


In [16]:
pos_J2000 = scb.ArrayWFrame(
    scb.ArrayWUnits([1, 2, 3], km),
    J2000
)

pos_ITRF93 = scb.ArrayWFrame(
    scb.ArrayWUnits([4, 5, 6], km),
    ITRF93
)

print("Same-frame addition:")
print(pos_J2000 + pos_J2000)

print("\nDifferent-frame addition:")

try:
    print(pos_J2000 + pos_ITRF93)
except Exception as e:
    print("Operation blocked:")
    print(e)


Same-frame addition:
[2. 4. 6.] [km ... km] (J2000 (0 - SOLAR SYSTEM BARYCENTER))

Different-frame addition:
Operation blocked:
('Frame J2000 (0 - SOLAR SYSTEM BARYCENTER) incompatible for mathematical operations with frame ', 'ITRF93 (399 - EARTH).')


## Summary

This notebook introduced:

- Dimensions and units
- `ArrayWUnits`
- SPICE frames
- Frame transformations
- `ArrayWFrame`
- Safe frame-aware operations

These abstractions are heavily used throughout Scarabaeus orbit determination,
navigation, estimation, and dynamics workflows.
